# Assignment 10: Forecasting El Niño with a convolutional network

El Niño and La Niña, the two phases of the El Niño–Southern Oscillation (ENSO), are the largest source of year-to-year climate variability on Earth. They shift rainfall, drought, hurricane activity and crop yields around the world, so a forecast made six months or a year ahead is valuable.

In 2019, Ham, Kim and Luo showed that a convolutional neural network given maps of ocean temperature could forecast ENSO more than a year ahead, with more skill than the physics-based forecast models of the time ([*Nature* 573, 568–572](https://doi.org/10.1038/s41586-019-1559-7)). This assignment builds a small version of their model. Their network also used ocean heat content and was trained on thousands of years of climate model output, so expect less skill here.

The task is **regression**. The input is a stack of three monthly maps of sea surface temperature (SST) anomaly over the tropical Pacific, and the output is one number: the Niño3.4 index some number of months later. The network treats the three months the way an image network treats the red, green and blue channels of a photo.

The data is the NOAA ERSST v5 field from Assignment 7, from 1880 to 2024. You will train on 1880 to 1969, validate on 1972 to 1989, test on 1992 to 2024, and compare the network with two baselines.

Answer each numbered question in the empty cell below it. Four questions ask for a written answer, and one sentence is enough for each.

In [ ]:
import os
import urllib.request

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dropout, Dense

from sklearn.linear_model import Ridge

**Data.** The cell below reads ERSST v5 from the class server, or downloads it (about 160 MB) if you are working somewhere else. It then does three things for you.

- It computes monthly anomalies relative to a climatology from the **training years only**, so nothing from the validation or test years leaks into the inputs.
- It subtracts the tropical mean anomaly from every grid cell, month by month, much as you subtracted the global mean in Assignment 7. The tropics have warmed since 1880, and without this step the later decades would look like one long El Niño.
- It selects the input region, 30S to 30N and 100E to 290E (a 31 × 96 grid), and computes the Niño3.4 index, the anomaly averaged over 5S to 5N and 190E to 240E.

In [ ]:
DATA = "/data9/CLMT5120/data/shr/sst.mnmean.nc"    # NOAA ERSST v5, copied to the class server
if not os.path.exists(DATA):                       # off the server: download a copy once
    DATA = "sst.mnmean.nc"
    if not os.path.exists(DATA):
        urllib.request.urlretrieve(
            "https://downloads.psl.noaa.gov/Datasets/noaa.ersst.v5/sst.mnmean.nc", DATA)

sst = xr.open_dataset(DATA)["sst"].sel(time=slice("1880", "2024")).load()

# Monthly anomalies, relative to the 1880 to 1969 mean for each calendar month
clim = sst.sel(time=slice("1880", "1969")).groupby("time.month").mean()
anom = sst.groupby("time.month") - clim

# Subtract the area-weighted tropical mean anomaly (20S to 20N) from every grid cell
tropics = anom.sel(lat=slice(20, -20))
anom = anom - tropics.weighted(np.cos(np.deg2rad(tropics.lat))).mean(["lat", "lon"])

# Inputs: 30S to 30N, 100E to 290E. Land has no SST, so land cells are set to 0 (no anomaly).
maps = anom.sel(lat=slice(30, -30), lon=slice(100, 290)).fillna(0.0)

# Target: the Nino3.4 index
nino34 = anom.sel(lat=slice(5, -5), lon=slice(190, 240)).mean(["lat", "lon"])

print(maps.shape, "= (months, latitudes, longitudes)")

## Part 1: Look at the data

1) Plot the Niño3.4 index from 1880 to 2024. Then plot the input map for **December 1997**, a strong El Niño, and for **December 1988**, a strong La Niña, side by side with a color scale centered on zero, and draw the Niño3.4 box on both.

## Part 2: Build the samples

2) Write a function `make_samples(lead)` that returns four arrays, with one entry for each starting month $t$:

- `X`, with shape (samples, 31, 96, 3): the maps for months $t-2$, $t-1$ and $t$, stacked as three channels
- `y`: the Niño3.4 index at month $t$ + `lead`, the target
- `persist`: the Niño3.4 index at month $t$, which the persistence baseline uses as its forecast
- `t`: the index of the starting month in `maps.time`

`maps.values` and `nino34.values` give plain NumPy arrays, and `np.stack([...], axis=-1)` stacks maps as channels. Call it with `lead=6` and print the shapes.

3) Split the samples by the year of the starting month: training 1880 to 1969, validation 1972 to 1989, test 1992 to 2024. The two-year gaps keep the periods apart. With a 12-month lead, a sample that starts in late 1969 has its target in late 1970, and that target must not fall in the validation period.

Then scale `X` by dividing it by its standard deviation over the training samples. Use one number for the whole array, so the anomalies keep their relative sizes across the map. Report the size of each split.

## Part 3: The CNN

4) Build this network with `Sequential`:

- `Input(shape=(31, 96, 3))`
- `Conv2D(16, 3, padding="same", activation="relu")`, then `MaxPooling2D(2)`
- `Conv2D(32, 3, padding="same", activation="relu")`, then `MaxPooling2D(2)`
- `Conv2D(32, 3, padding="same", activation="relu")`, then `MaxPooling2D(2)`
- `Flatten()`, `Dropout(0.3)` and `Dense(32, activation="relu")`
- `Dense(1)` with no activation, since the output is a number rather than a class

Print the summary and report the number of trainable parameters. **Written answer (one sentence):** which layer holds the most parameters, and why do the convolutional layers need so few?

5) Compile with mean squared error loss and `keras.optimizers.Adam(learning_rate=3e-4)`. Call `keras.utils.set_random_seed(0)`, then train for up to 300 epochs with `batch_size=32`, using the validation set and an `EarlyStopping` callback (`patience=20`, `restore_best_weights=True`). Plot the training and validation loss and report the epoch with the lowest validation loss.

## Part 4: Compare with baselines

6) On the test set, compute the correlation and the root mean squared error (RMSE) between the CNN's predictions and the observed index. Do the same for two baselines:

- **Persistence**: the forecast for month $t$ + `lead` is the value at month $t$, `persist`.
- **Ridge regression** on the same maps, flattened with `X.reshape(len(X), -1)`. Choose `alpha` from 10, 100, 1000, 10,000 and 100,000 by the RMSE on the validation set.

Show the six numbers in one table.

7) Plot the observed index over the test period together with the CNN and persistence forecasts, with each forecast placed at its **target** month. **Written answer (one sentence):** what does persistence get wrong at a six-month lead?

8) Networks trained from different random starts give different answers, as you saw in Week 10. Train four more CNNs at a six-month lead with seeds 1 to 4. Report the test correlation of each of the five, and the correlation of their average prediction.

## Part 5: How far ahead?

9) Repeat the comparison for leads of 1, 3, 6, 9 and 12 months. For each lead, rebuild and scale the samples, train one CNN (seed 0), fit ridge regression with `alpha` chosen on the validation set, and compute persistence. Plot the test correlation of all three against lead. Expect the CNN's numbers to move by about 0.05 from one run to the next. **Written answer (one sentence):** compare how quickly skill falls with lead for persistence and for the two models that see the maps.

10) ENSO forecasts are known to be hardest across the northern spring, the **spring predictability barrier**. At a six-month lead, compute the test correlation separately for each calendar month of the starting month $t$, for the CNN from question 5 and for persistence, and show them as a bar chart. **Written answer (one sentence):** which starting months give the weakest forecasts?

## What to hand in

The notebook with all cells run, and the four one-sentence written answers (questions 4, 7, 9 and 10) in their markdown cells.